In [ ]:
import re
import pandas as pd

def normalize_text(s):
    """Normaliza texto (sem acentos, minúsculo, sem caracteres estranhos)."""
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9x ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def extract_numbers_and_units(s):
    """
    Extrai números e checa se são seguidos de unidade (ml, mg, g, ui).
    Retorna tuplas (valor, tem_unidade).
    """
    if pd.isna(s):
        return []
    s = str(s).lower()
    matches = re.findall(r'(\d+(?:[\.,]\d+)?)\s*(ml|mg|g|ui)?', s)
    result = []
    for num, unit in matches:
        num = float(num.replace(',', '.'))
        result.append((num, bool(unit)))
    return result

def choose_candidate_pack(text):
    """
    Decide qual número usar como pack-size:
    - ignora números com unidade (ml, mg, g, ui)
    - considera apenas inteiros de 2 a 500
    - se tiver vários, pega o maior (ex: '3x50' → 50).
    """
    nums = extract_numbers_and_units(text)
    candidates = [int(round(n)) for n, has_unit in nums if not has_unit and 2 <= n <= 500]
    if not candidates:
        return 0
    return max(candidates)

def find_times_pattern(s):
    """Procura padrões tipo '3x50' ou '3 x 50' e retorna (3, 50)."""
    if pd.isna(s):
        return None
    s = str(s).lower()
    m = re.search(r'(\d{1,4})\s*[x×]\s*(\d{1,4})', s)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None

def heuristic_multiplier(material, nomeconc):
    """
    Heurística:
    - Se achar '3x50', usa o 3 como multiplicador e 50 como pack.
    - Se achar c/30 → pack = 30.
    - Ignora números com unidade (ml, mg, g, ui).
    """
    m_pack = choose_candidate_pack(material)
    n_pack = choose_candidate_pack(nomeconc)

    m_times = find_times_pattern(material)
    n_times = find_times_pattern(nomeconc)

    multiplier = 1.0
    pack = max(m_pack, n_pack, 0)

    if m_times:
        multiplier = m_times[0]
        pack = max(pack, m_times[1])
    elif n_times:
        multiplier = n_times[0]
        pack = max(pack, n_times[1])
    else:
        # se nomeconc tem pack e material não → provavelmente divide
        if n_pack and not m_pack:
            multiplier = 1.0 / n_pack
        elif m_pack and not n_pack:
            multiplier = 1.0
    return multiplier, pack


# ----------------------------
# EXEMPLOS DE TESTE
# ----------------------------
dados = [
    ("VITAMINA C 1000MG C/30 COMP", "VITAMINA C 1000MG", "esperado divide por 30"),
    ("SHAMPOO 200ML C/12", "SHAMPOO 200ML", "esperado multiplica 12"),
    ("SUPLEMENTO 3X50 CAPS", "SUPLEMENTO C/50 CAPS", "esperado multiplica 3"),
    ("CREME HIDRATANTE 200ML", "CREME HIDRATANTE 200ML", "esperado fator ~1"),
]

for mat, nome, esperado in dados:
    mult, pack = heuristic_multiplier(mat, nome)
    print(f"MATERIAL: {mat}")
    print(f"NOME_CONC: {nome}")
    print(f"Resultado: mult={mult}, pack={pack} | {esperado}")
    print("-"*50)
